In [ ]:
import requests
from bs4 import BeautifulSoup
import re

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import time

def get_cdn_urls_test(search_page_html, limit=10):
    # 1. 기존 ipynb 방식: HTML 전체에서 playId 추출
    soup = BeautifulSoup(search_page_html, "html.parser")

    # 비디오 아이콘이 포함된 모든 하이퍼링크 찾기
    links = soup.find_all("a", href=re.compile(r"playId="))

    cdn_list = []

    # 2. 테스트를 위해 상위 10개만 루프 실행
    for idx, link in enumerate(links[:limit]):
        href = link.get("href")
        play_id = href.split("playId=")[-1].split("&")[0]

        # 3. 개별 영상 페이지 접속 (기존 코드 방식)
        video_page_url = f"https://baseballsavant.mlb.com/sporty-videos?playId={play_id}"

        try:
            res_p2 = requests.get(video_page_url)
            soup_p2 = BeautifulSoup(res_p2.text, "html.parser")

            # 4. <video> 태그 안의 실제 CDN 주소(src) 따오기
            video_tag = soup_p2.find("video")
            if video_tag and video_tag.find("source"):
                cdn_url = video_tag.find("source").get("src")
                print(f"[{idx+1}] PlayID: {play_id[:8]}... -> CDN 주소 확보 성공")
                cdn_list.append(cdn_url)
            else:
                print(f"[{idx+1}] {play_id[:8]}... -> CDN 주소를 찾을 수 없음 (페이지 로딩 방식 변경 가능성)")

            time.sleep(1) # 서버 부하 방지

        except Exception as e:
            print(f"[{idx+1}] 접속 중 에러 발생: {e}")

    return cdn_list


In [ ]:
## 2025년 정규 시즌 전체 투구 수 영상데이터 전부 불러오기 -> 마지막에 player_id=119 (팀 아이디) 넣어주면 됨.
# 선수별로 안해도 되니까 하나의 주소만 가지고 쭉 불러오면 됨.

url2 = f'https://baseballsavant.mlb.com/statcast_search?hfPT=&hfAB=&hfGT=R%7C&hfPR=&hfZ=&hfStadium=&hfBBL=&hfNewZones=&hfPull=&hfC=&hfSea=2025%7C&hfSit=&player_type=pitcher&hfOuts=&home_road=&pitcher_throws=&batter_stands=&hfSA=&hfEventOuts=&hfEventRuns=&game_date_gt=&game_date_lt=&hfMo=&hfTeam=119%7C&hfOpponent=&hfRO=&position=&hfInfield=&hfOutfield=&hfInn=&hfBBT=&hfFlag=&metric_1=&group_by=team&min_pitches=0&min_results=0&min_pas=0&sort_col=pitches&player_event_sort=api_p_release_speed&sort_order=desc&type=details&player_id=119'
res = requests.get(url2)
soup = BeautifulSoup(res.text, "html.parser")
video_tags = soup.find_all('a')
play_ids = []
for tag in video_tags:
        href = tag.get('href') # 전체 주소를 가져옴
        if href:
            print(href)
            # 2. 정규표현식으로 playId= 뒤의 UUID 부분만 추출
            # [0-9a-f-] 패턴은 UUID에 사용되는 16진수와 하이픈을 의미합니다.
            match = re.search(r'playId=([0-9a-f-]{36})', href)

            if match:
                uuid = match.group(1)
                play_ids.append(uuid)
                print(f"찾은 UUID: {uuid}")
print(play_ids)

/sporty-videos?playId=6d318a04-d3e6-3d5e-96ef-1e4cd61c7a4b
찾은 UUID: 6d318a04-d3e6-3d5e-96ef-1e4cd61c7a4b
/sporty-videos?playId=d28c4926-db7c-33c6-8f22-6ac9812506ed
찾은 UUID: d28c4926-db7c-33c6-8f22-6ac9812506ed
/sporty-videos?playId=77e6f955-3f16-3cb2-9ee5-a095fa99d7f1
찾은 UUID: 77e6f955-3f16-3cb2-9ee5-a095fa99d7f1
/sporty-videos?playId=7e41ddc2-0fe5-3de3-be86-626ac4e9f543
찾은 UUID: 7e41ddc2-0fe5-3de3-be86-626ac4e9f543
/sporty-videos?playId=890a0bfe-3712-321c-8954-0ece441ba49f
찾은 UUID: 890a0bfe-3712-321c-8954-0ece441ba49f
/sporty-videos?playId=1583ca46-2212-3597-973c-f5d0de88077f
찾은 UUID: 1583ca46-2212-3597-973c-f5d0de88077f
/sporty-videos?playId=b8ea7edc-243c-33a6-b2b1-06d2dcfac848
찾은 UUID: b8ea7edc-243c-33a6-b2b1-06d2dcfac848
/sporty-videos?playId=8b0d13f6-ac40-3ee7-ac75-cfad78ca36a0
찾은 UUID: 8b0d13f6-ac40-3ee7-ac75-cfad78ca36a0
/sporty-videos?playId=057f2148-e6d5-3dc3-b2e6-9274bc17a41d
찾은 UUID: 057f2148-e6d5-3dc3-b2e6-9274bc17a41d
/sporty-videos?playId=e9c65d0f-378f-3743-a4a2-71bf0d9b6

In [ ]:
len(play_ids)

24073

In [ ]:
## 선수별로 영상데이터 불러오기
# player_ids = [808967, 605483, 686218, 607192, 680736, 660271,
#               808963, 595014, 607455, 683618, 676263, 694361, 477132, 681911, 689017, 571760, 656945, 489446, 656629, 676508, 669422, 801434, 664747, 592779, 669160, 642152, 642759, 669193,
#               500743, 663562, 472610, 571771, 608717, 570632, 702795, 664062, 676272, 656420, 672782, 623465, 689233, 814527, 669188, 672277, 692064, 701328]
player_ids = [605483]

video_tags = []
play_ids = []
for player_id in player_ids:
    url2 = f'https://baseballsavant.mlb.com/statcast_search?hfPT=FF%7C&hfAB=&hfGT=R%7C&hfPR=&hfZ=&hfStadium=&hfBBL=&hfNewZones=&hfPull=&hfC=&hfSea=2025%7C&hfSit=&player_type=pitcher&hfOuts=&home_road=&pitcher_throws=&batter_stands=&hfSA=&hfEventOuts=&hfEventRuns=&game_date_gt=&game_date_lt=&hfMo=&hfTeam=119%7C&hfOpponent=&hfRO=&position=&hfInfield=&hfOutfield=&hfInn=&hfBBT=&hfFlag=&pitchers_lookup%5B%5D={player_id}&metric_1=&group_by=name&min_pitches=0&min_results=0&min_pas=0&sort_col=pitches&player_event_sort=api_p_release_speed&sort_order=desc&type=details&player_id={player_id}'
    res = requests.get(url2)
    soup = BeautifulSoup(res.text, "html.parser")
    #video_tags.append( soup.find_all('a'))
    video_tags = soup.find_all('a')

    for tag in video_tags:
        href = tag.get('href') # 전체 주소를 가져옴
        if href:
            print(href)
            # 2. 정규표현식으로 playId= 뒤의 UUID 부분만 추출
            # [0-9a-f-] 패턴은 UUID에 사용되는 16진수와 하이픈을 의미합니다.
            match = re.search(r'playId=([0-9a-f-]{36})', href)

            if match:
                uuid = match.group(1)
                play_ids.append(uuid)
                print(f"찾은 UUID: {uuid}")

print(f"총 {len(play_ids)}개의 ID를 찾았습니다.")

/sporty-videos?playId=3c2afecf-a864-31b7-8baa-ccab401c4a5a
찾은 UUID: 3c2afecf-a864-31b7-8baa-ccab401c4a5a
/sporty-videos?playId=a3318fab-39f2-3d89-80f5-a7453ec6a1b5
찾은 UUID: a3318fab-39f2-3d89-80f5-a7453ec6a1b5
/sporty-videos?playId=ca5adfe1-a19b-3eac-998c-3a1414683200
찾은 UUID: ca5adfe1-a19b-3eac-998c-3a1414683200
/sporty-videos?playId=d4e747fe-fc43-3c6f-ba59-878736ab9500
찾은 UUID: d4e747fe-fc43-3c6f-ba59-878736ab9500
/sporty-videos?playId=39a46b75-8270-3e9a-bf34-551a42bccdf9
찾은 UUID: 39a46b75-8270-3e9a-bf34-551a42bccdf9
/sporty-videos?playId=0c6e6a1c-7e0e-3a66-a66d-1a4775962ac3
찾은 UUID: 0c6e6a1c-7e0e-3a66-a66d-1a4775962ac3
/sporty-videos?playId=7d22bc03-e13a-3cd8-95a2-86ac32177569
찾은 UUID: 7d22bc03-e13a-3cd8-95a2-86ac32177569
/sporty-videos?playId=5ea6599a-5273-3bb9-9349-8a0aca925283
찾은 UUID: 5ea6599a-5273-3bb9-9349-8a0aca925283
/sporty-videos?playId=6e504eb3-d41d-318b-ad61-485d8d951347
찾은 UUID: 6e504eb3-d41d-318b-ad61-485d8d951347
/sporty-videos?playId=3bf2ef80-80d0-3465-be3e-f44e56ad3

In [ ]:
def download_video(url, filename):
    # 브라우저인 것처럼 보이기 위한 헤더 설정
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        # stream=True 옵션으로 연결을 유지하며 조각씩 다운로드
        with requests.get(url, headers=headers, stream=True) as r:
            r.raise_for_status()  # 접속 에러 발생 시 예외 처리

            with open(filename, 'wb') as f:
                print(f"다운로드 시작: {filename}")
                # 8KB 단위로 조각내서 파일에 기록
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        print(f"다운로드 완료!")
    except Exception as e:
        print(f"다운로드 중 오류 발생: {e}")

# 실행
# video_url = res_p2.find("video").find("source").get("src")
# download_video(video_url, "mlb_pitch_video.mp4")

In [ ]:
# 1. 태그가 'a'이면서 href에 '/sporty-videos'가 포함된 모든 요소 찾기
video_tags = soup.find_all('a')

play_ids = []

for tag in video_tags:
    href = tag.get('href') # 전체 주소를 가져옴
    if href:
        print(href)
        # 2. 정규표현식으로 playId= 뒤의 UUID 부분만 추출
        # [0-9a-f-] 패턴은 UUID에 사용되는 16진수와 하이픈을 의미합니다.
        match = re.search(r'playId=([0-9a-f-]{36})', href)

        if match:
            uuid = match.group(1)
            play_ids.append(uuid)
            print(f"찾은 UUID: {uuid}")

# 결과 확인
print(f"총 {len(play_ids)}개의 ID를 찾았습니다.")

/sporty-videos?playId=3c2afecf-a864-31b7-8baa-ccab401c4a5a
찾은 UUID: 3c2afecf-a864-31b7-8baa-ccab401c4a5a
/sporty-videos?playId=a3318fab-39f2-3d89-80f5-a7453ec6a1b5
찾은 UUID: a3318fab-39f2-3d89-80f5-a7453ec6a1b5
/sporty-videos?playId=ca5adfe1-a19b-3eac-998c-3a1414683200
찾은 UUID: ca5adfe1-a19b-3eac-998c-3a1414683200
/sporty-videos?playId=d4e747fe-fc43-3c6f-ba59-878736ab9500
찾은 UUID: d4e747fe-fc43-3c6f-ba59-878736ab9500
/sporty-videos?playId=39a46b75-8270-3e9a-bf34-551a42bccdf9
찾은 UUID: 39a46b75-8270-3e9a-bf34-551a42bccdf9
/sporty-videos?playId=0c6e6a1c-7e0e-3a66-a66d-1a4775962ac3
찾은 UUID: 0c6e6a1c-7e0e-3a66-a66d-1a4775962ac3
/sporty-videos?playId=7d22bc03-e13a-3cd8-95a2-86ac32177569
찾은 UUID: 7d22bc03-e13a-3cd8-95a2-86ac32177569
/sporty-videos?playId=5ea6599a-5273-3bb9-9349-8a0aca925283
찾은 UUID: 5ea6599a-5273-3bb9-9349-8a0aca925283
/sporty-videos?playId=6e504eb3-d41d-318b-ad61-485d8d951347
찾은 UUID: 6e504eb3-d41d-318b-ad61-485d8d951347
/sporty-videos?playId=3bf2ef80-80d0-3465-be3e-f44e56ad3

In [ ]:
for idx in range(len(play_ids)):
    url_p2 = f"https://baseballsavant.mlb.com/sporty-videos?playId={play_ids[idx]}"
    res_p2 = requests.get(url_p2)
    res_p2 = BeautifulSoup(res_p2.text,"html.parser")
    res_p2.find("video").find("source").get("src")
    video_url = res_p2.find("video").find("source").get("src")
    download_video(video_url, f"mlb_pitch_video{idx}.mp4")

# 실행

다운로드 시작: mlb_pitch_video0.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video1.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video2.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video3.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video4.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video5.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video6.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video7.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video8.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video9.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video10.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video11.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video12.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video13.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video14.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video15.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video16.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video17.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video18.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video19.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video20.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video21.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video22.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video23.mp4
다운로드 완료!
다운로드 시작: mlb_pitch_video24.mp4
다운로드 완료!
다운로드 시작: m